# 🜏 SOV33 Sovereign Merge — Colab Runbook STEP 2

**Goal:** Prove the sovereign-merge architecture on Qwen3.6-4B base. GATE 1 verdict: does the merged sovereign model beat the base on the 65-task real held-out governance battery?

**Time:** 2-3 hours wall-clock.
**Cost:** $0 (Colab free tier, T4 16GB).
**Output:** Charter-1 sovereign model + benchmark verdict.

## Before you run

1. Open https://colab.research.google.com/ in your browser
2. Sign in with your Google account
3. Runtime → Change runtime type → **T4 GPU** (free tier, 16GB VRAM)
4. Upload this notebook (or copy cells one by one)
5. Run each cell in order

## What this proves

* Sovereign merge → 4 fine-tuned experts (COMPLIANCE / DEFENSE / INTUITION / VOICE)
* mergekit TIES merge → 1 sovereign model
* Real held-out battery → 65 governance tasks (derived from the same on-disk data the experts were trained on, with deterministic held-out split)
* **GATE 1:** merged model pass rate vs base pass rate

## What gets exported

* Charter-1 sovereign merge weights (LoRA merged into base)
* Benchmark results JSON
* SIGIL-signed audit digest of the run (timestamped)
* Result: the sovereign-by-construction model is on disk, ready for the next step (Charter-2 on 35B, or Charter-Ω sovereign merge v0.3)

## STEP 1: Install the stack

In [ ]:
!pip install -q "transformers>=4.44" peft trl bitsandbytes accelerate datasets mergekit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
print(f"torch: {torch.__version__}, cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory // 1_000_000_000}GB)")
    print(f"GPU compute capability: {torch.cuda.get_device_capability(0)}")

## STEP 2: Verify the GPU is real and can hold the 4B model with QLoRA

In [ ]:
import torch
assert torch.cuda.is_available(), "Need a GPU. Runtime → Change runtime type → T4 GPU"
assert torch.cuda.get_device_properties(0).total_memory >= 16 * 1_000_000_000, "Need ≥16GB VRAM"
print(f"✓ GPU OK: {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✓ T4 can fit Qwen3.6-4B in 4-bit QLoRA (~8GB VRAM used)")

## STEP 3: Clone the sovereign-merge kit from CSOAI-ORG

In [ ]:
!git clone https://github.com/CSOAI-ORG/clawd-workspace.git /content/clawd 2>&1 | tail -3
%cd /content/clawd
import os
os.chdir('/content/clawd')
print(f"CWD: {os.getcwd()}")
import subprocess
subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True)

## STEP 4: Run the data prep (3,926 sovereign-labelled examples)

In [ ]:
!cd /content/clawd/_alignment/sovereign_merge_kit && python3 01_prep_expert_data.py 2>&1 | tail -10

## STEP 5: Build the real held-out benchmark (65 tasks, deterministic)

In [ ]:
!cd /content/clawd/_alignment/sovereign_merge_kit && python3 04_benchmark_REAL.py --build 2>&1 | tail -5

## STEP 6: Fine-tune 4 sovereign experts (QLoRA, ~30 min each = ~2 hours total)

In [ ]:
import subprocess, time
experts = ['compliance', 'defense', 'intuition', 'voice']
for expert in experts:
    print(f"\n{'='*60}\n  Fine-tuning {expert}\n{'='*60}")
    start = time.time()
    r = subprocess.run(
        ['python3', '02_finetune_expert.py',
         '--expert', expert,
         '--base', 'Qwen/Qwen3.6-4B',
         '--data', f'expert_data/{expert}.jsonl',
         '--epochs', '2.0'],
        capture_output=True, text=True, timeout=3600,
    )
    elapsed = (time.time() - start) / 60
    if r.returncode == 0:
        print(f"  ✓ {expert}: trained in {elapsed:.1f} min")
    else:
        print(f"  ✗ {expert}: error after {elapsed:.1f} min")
        print(r.stderr[-500:])

## STEP 7: Merge the 4 sovereign experts via mergekit TIES

In [ ]:
!mergekit-yaml 03_merge_experts.yaml ./charter-1 --allow-crimes 2>&1 | tail -10
!ls -la charter-1/ 2>&1 | head -10

## STEP 8: GATE 1 — Run the 65-task real held-out benchmark

In [ ]:
!python3 04_benchmark_REAL.py \
  --models base=Qwen/Qwen3.6-4B merged=./charter-1 \
  --data-dir expert_data/ 2>&1 | tail -20

## STEP 9: SIGIL-signed audit digest of the run

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives import serialization

# Build the audit record
audit_record = {
    "ts": datetime.now(timezone.utc).isoformat(),
    "phase": "STEP 2 — Sovereign Merge v0.1 on Colab free tier",
    "base": "Qwen/Qwen3.6-4B",
    "frozen_lora": True,
    "experts": ["compliance", "defense", "intuition", "voice"],
    "merge_method": "mergekit-yaml TIES",
    "battery": "65 real held-out governance tasks (deterministic split)",
    "verdict": "See STEP 8 output above",
    "weight_dir": "/content/clawd/_alignment/sovereign_merge_kit/charter-1/",
}

# Generate the sovereign key (one-time, persists for the duration of this Colab session)
sovereign_key = Ed25519PrivateKey.generate()
sovereign_pub = sovereign_key.public_key().public_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PublicFormat.Raw,
)

# Sign the audit record
payload = json.dumps(audit_record, sort_keys=True).encode()
digest = hashlib.sha256(payload).hexdigest()[:16]
signature = sovereign_key.sign(payload)

# Export the audit
audit_full = {**audit_record, "audit": {
    "digest": digest,
    "ed25519_pubkey": sovereign_pub.hex(),
    "ed25519_signature": signature.hex(),
    "note": "Quick sovereign-key demo. Production uses the canonical King key in ~/.sovereign/",
}}
with open('charter-1/COLAB_AUDIT.json', 'w') as f:
    json.dump(audit_full, f, indent=2)

print(f"✓ Audit digest: {digest}")
print(f"✓ Sovereign pubkey: {sovereign_pub.hex()[:32]}...")
print(f"✓ Audit saved to charter-1/COLAB_AUDIT.json")
print(f"\nGATE 1 verdict (above in STEP 8) is the GATE; this audit is the proof-of-provenance.")

## STEP 10: Save the run artifacts (sovereign weights + audit) to disk or Drive

In [ ]:
# Option A: zip and download
!cd /content/clawd/_alignment/sovereign_merge_kit && \
  tar czf charter-1.tar.gz charter-1/
!ls -lah charter-1.tar.gz

# Option B (optional): save to Google Drive if linked
# from google.colab import drive
# drive.mount('/content/drive')
# !cp charter-1.tar.gz /content/drive/MyDrive/

print("\n✓ Charter-1 sovereign merge artifacts ready")
print("  Download: charter-1.tar.gz")
print(f"  Audit: charter-1/COLAB_AUDIT.json (SIGIL-signed with Ed25519)")
print(f"\nNext move (Step 3): bring Charter-1 back to Vast.ai or local Qwen3.6-35B-A3B to scale to Charter-2 (the real base).")